In [ ]:
import cv2
import json
import os
import numpy as np
import pandas as pd
# Note: First time importing pytesseract/easyocr will take extra time
import pytesseract 
import easyocr

In [ ]:
IN_DIR = "../data/raw"
IN_FILE = "sample.png"
IN_PATH = os.path.join(IN_DIR, IN_FILE)
OUT_DIR = "../output"
CSV_FILE = "sample_extracted.csv"
CONFIG_DIR = "../configs"
JSON_FILE = "template.json"
OUT_PATH = os.path.join(OUT_DIR, CSV_FILE)
CONFIG_PATH = os.path.join(CONFIG_DIR, JSON_FILE)

In [ ]:
img = cv2.imread(IN_PATH)
cfg = json.load(open(CONFIG_PATH))

In [ ]:
def preprocess(cell, scale=3):
    resize = cv2.resize(cell, None, fx=scale, fy=scale, interpolation=cv2.INTER_CUBIC)
    gray = cv2.cvtColor(resize, cv2.COLOR_BGR2GRAY)
    _, thr = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    if np.mean(thr) < 127:
        thr = cv2.bitwise_not(thr)
    return thr

In [ ]:
def read_number(cell):
    c = "--psm 7 -c tessedit_char_whitelist=0123456789,"
    return pytesseract.image_to_string(preprocess(cell), config=c).strip().replace(',', '')

In [ ]:
def read_name(cell):
    reader = easyocr.Reader(["en"], gpu=True)
    res = reader.readtext(cell, detail=0)
    return res

In [ ]:
def extract_team(img, cfg, team):
    h = cfg["row_h"]
    nx1, nx2 = cfg["name_x"]
    rows = []
    for top in cfg["teams"][team]["row_tops"]:
        rec = {"team": team,
               "name": read_name(img[top:top+h, nx1:nx2])}
        for col, (x1, x2) in cfg["col_x"].items():
            rec[col] = read_number(img[top:top+h, x1:x2])
        rows.append(rec)
    return rows

In [ ]:
records = extract_team(img, cfg, "blue") + extract_team(img, cfg, "red")
df = pd.DataFrame(records)
df # Note: Reading 0% Ultimate from some cell names and Missing some empty zero 0 values for cell numbers

In [ ]:
df.to_csv(OUT_PATH, index=False)
print(f"wrote {OUT_PATH}")